# YOLOv8L — Indian Grocery Supercategory Fine-tuning

Fine-tunes a pre-trained RPC YOLOv8L model on the Indian Grocery dataset with 5 supercategories.

**Research Pipeline:** YOLO (supercategory) → ViT (brand) → OCR (validation)

In [ ]:
!pip install -U ultralytics pyyaml

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["YOLO_DISABLE_JPEG_REPAIR"] = "1"
os.environ["YOLO_VERBOSE"] = "false"

## Step 1: Label Conversion — Brand → Supercategory

The Indian Grocery dataset has 10 brand-level classes. We remap them to 5 supercategories matching our `data.yaml`.

| Brand | Supercategory | ID |
|---|---|---|
| Bournvita, Nescafe, Society Tea | beverage | 0 |
| (reserved for packed food) | food-box | 1 |
| (reserved) | fruit | 2 |
| Soap, Lotion, Cream, Oil, Toothpaste, Shampoo, Conditioner | utility-box | 3 |
| (reserved) | vegetable | 4 |

In [ ]:
from pathlib import Path

# Original 10 brand classes from the downloaded dataset (index = class_id in label files)
original_classes = [
    "Bournvita",                              # 0
    "Mysore Sandal Soap",                     # 1
    "Nescafe_Classic_Coffee",                 # 2
    "Nivea Body Lotion",                      # 3
    "Nivea_Soft_Moisturising_cream",          # 4
    "Parachute coconut Oil",                  # 5
    "Patanjali Dant Kanti",                   # 6
    "Society_TeaPowder_plain",                # 7
    "Tresemme_Hairfall_Defense_conditioner",  # 8
    "Tresemme_Hairfall_Defense_shampoo",      # 9
]

# Brand → supercategory mapping
fine_to_super = {
    "Bournvita":                              "beverage",
    "Nescafe_Classic_Coffee":                 "beverage",
    "Society_TeaPowder_plain":                "beverage",
    "Mysore Sandal Soap":                     "utility-box",
    "Patanjali Dant Kanti":                   "utility-box",
    "Nivea Body Lotion":                      "utility-box",
    "Nivea_Soft_Moisturising_cream":          "utility-box",
    "Parachute coconut Oil":                  "utility-box",
    "Tresemme_Hairfall_Defense_conditioner":  "utility-box",
    "Tresemme_Hairfall_Defense_shampoo":      "utility-box",
}

# Supercategory ID — MUST match data.yaml names list order
super_to_id = {
    "beverage":    0,
    "food-box":    1,
    "fruit":       2,
    "utility-box": 3,
    "vegetable":   4,
}

id_to_class = {i: name for i, name in enumerate(original_classes)}

print("Supercategory ID mapping:")
for k, v in super_to_id.items():
    brands = [b for b, s in fine_to_super.items() if s == k]
    print(f"  {v}: {k} → {brands if brands else "(no brands in this dataset)"}")

In [ ]:
# ── Convert all YOLO label files from brand-level to supercategory-level ──────
def convert_labels(label_dir):
    label_dir = Path(label_dir)
    if not label_dir.exists():
        print(f"  SKIPPED (not found): {label_dir}")
        return

    converted = skipped = 0
    for file in label_dir.glob("*.txt"):
        with open(file, "r") as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            class_id   = int(parts[0])
            class_name = id_to_class.get(class_id)
            super_name = fine_to_super.get(class_name)
            if class_name is None or super_name is None:
                skipped += 1
                continue
            new_class_id = super_to_id[super_name]
            new_lines.append(" ".join([str(new_class_id)] + parts[1:]))
            converted += 1

        with open(file, "w") as f:
            f.write("\n".join(new_lines))

    print(f"  {label_dir.name}: {converted} annotations converted, {skipped} skipped")


# ── Set DATASET_ROOT to your local or Kaggle path ───────────────────────────
# Kaggle: DATASET_ROOT = Path("/kaggle/input/indian-grocery-object-detection")
DATASET_ROOT = Path.home() / "Downloads" / "Indian Grocery Object Detection.v1i.yolov8"

print("Converting labels to supercategory IDs...")
convert_labels(DATASET_ROOT / "train" / "labels")
convert_labels(DATASET_ROOT / "valid" / "labels")
convert_labels(DATASET_ROOT / "test"  / "labels")  # no-op if test split missing
print("\n✅ Label conversion complete!")

## Step 2: Write data.yaml

In [ ]:
import yaml

# On Kaggle: DATASET_ROOT = Path("/kaggle/input/indian-grocery-object-detection")
DATASET_ROOT = Path.home() / "Downloads" / "Indian Grocery Object Detection.v1i.yolov8"

data_yaml_content = {
    "train": str(DATASET_ROOT / "train" / "images"),
    "val":   str(DATASET_ROOT / "valid" / "images"),
    "nc":    5,
    "names": ["beverage", "food-box", "fruit", "utility-box", "vegetable"],
}

yaml_path = DATASET_ROOT / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)

print(f"data.yaml written to: {yaml_path}")
print()
with open(yaml_path) as f:
    print(f.read())

## Step 3: Fine-tune RPC Model on Indian Grocery Dataset

In [ ]:
import time
import pandas as pd
from ultralytics import YOLO

epoch_times = []

def on_train_start(trainer):
    trainer._epoch_start_time = time.time()
    print("Fine-tuning started: RPC weights → Indian Grocery supercategories\n")

def on_train_epoch_start(trainer):
    trainer._epoch_start_time = time.time()

def on_train_epoch_end(trainer):
    epoch_time = time.time() - trainer._epoch_start_time
    epoch_times.append(epoch_time)
    avg_epoch = sum(epoch_times) / len(epoch_times)
    remaining = trainer.epochs - (trainer.epoch + 1)
    eta = int(avg_epoch * remaining)
    h, rem = divmod(eta, 3600)
    m, s   = divmod(rem, 60)
    mtr   = trainer.metrics
    box   = mtr.get("val/box_loss",        float("nan"))
    cls   = mtr.get("val/cls_loss",        float("nan"))
    dfl   = mtr.get("val/dfl_loss",        float("nan"))
    map50 = mtr.get("metrics/mAP50(B)",    float("nan"))
    print(
        f"Epoch {trainer.epoch+1}/{trainer.epochs} | "
        f"Time: {epoch_time:.1f}s | ETA: {h:02d}:{m:02d}:{s:02d} | "
        f"box: {box:.4f} | cls: {cls:.4f} | dfl: {dfl:.4f} | mAP50: {map50:.4f}"
    )

In [ ]:
import json

# ── Model path: your RPC-trained best.pt ────────────────────────────────────
# Kaggle: add your saved RPC model as a dataset input
# Example: RPC_MODEL = "/kaggle/input/rpc-yolo-trained/best.pt"
RPC_MODEL = "/kaggle/working/best.pt"  # ← update if needed

DATA_YAML = str(DATASET_ROOT / "data.yaml")

EXPERIMENTS = {
    "indian_grocery_finetune": {
        "name":        "Indian Grocery Fine-tune",
        "description": "Transfer RPC weights → 5-class Indian grocery supercategories",
        "model":       RPC_MODEL,
        "data_yaml":   DATA_YAML,
        "epochs":      30,
        "imgsz":       640,
        "batch":       8,
        "mosaic":      0.5,
        "copy_paste":  0.1,
        "lr0":         1e-4,  # Low LR for fine-tuning
    }
}


def run_experiment(exp_name, config):
    print("=" * 70)
    print(f"EXPERIMENT: {config["name"]}")
    print("=" * 70)
    print(f"Description: {config["description"]}\n")

    model = YOLO(config["model"])
    model.add_callback("on_train_start",       on_train_start)
    model.add_callback("on_train_epoch_start", on_train_epoch_start)
    model.add_callback("on_train_epoch_end",   on_train_epoch_end)

    results = model.train(
        data         = config["data_yaml"],
        epochs       = config["epochs"],
        imgsz        = config["imgsz"],
        batch        = config["batch"],
        workers      = 2,
        cache        = False,
        rect         = False,
        optimizer    = "AdamW",
        lr0          = config["lr0"],
        lrf          = 0.01,
        weight_decay = 5e-4,
        mosaic       = config["mosaic"],
        copy_paste   = config["copy_paste"],
        close_mosaic = 5,
        freeze       = 10,   # Freeze first 10 layers — preserves RPC backbone features
        amp          = True,
        save         = True,
        save_period  = 5,
        plots        = False,
        verbose      = True,
    )

    csv_path = Path(model.trainer.save_dir) / "results.csv"
    df       = pd.read_csv(csv_path)
    last     = df.iloc[-1]
    print(
        f"\nFINAL EPOCH {int(last["epoch"])}\n"
        f"mAP50      : {last["metrics/mAP50(B)"]:.4f}\n"
        f"mAP50-95   : {last["metrics/mAP50-95(B)"]:.4f}\n"
        f"Precision  : {last["metrics/precision(B)"]:.4f}\n"
        f"Recall     : {last["metrics/recall(B)"]:.4f}"
    )
    return last.to_dict()


def main():
    all_results = {}
    for exp_name, config in EXPERIMENTS.items():
        all_results[exp_name] = run_experiment(exp_name, config)
    with open("/kaggle/working/indian_grocery_results.json", "w") as f:
        json.dump(all_results, f, indent=2)
    print("\n TRAINING COMPLETE")


if __name__ == "__main__":
    main()

In [ ]:
import shutil

run_dir = Path("runs/detect")
latest  = sorted(run_dir.glob("train*"))[-1]

shutil.copy(latest / "weights/best.pt", "/kaggle/working/indian_grocery_best.pt")
shutil.copy(latest / "weights/last.pt", "/kaggle/working/indian_grocery_last.pt")

print("Models saved:")
print("  /kaggle/working/indian_grocery_best.pt  ← use this in ViT-OCR notebook")
print("  /kaggle/working/indian_grocery_last.pt")

## Step 4: Quick Inference Test

In [ ]:
import cv2
import matplotlib.pyplot as plt

YOLO_MODEL_PATH = "/kaggle/working/indian_grocery_best.pt"
yolo = YOLO(YOLO_MODEL_PATH)

TEST_DIR    = DATASET_ROOT / "valid" / "images"
test_images = list(TEST_DIR.glob("*.jpg"))[:3]

for img_path in test_images:
    results    = yolo(str(img_path), conf=0.3)[0]
    annotated  = results.plot()
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title(img_path.name)
    plt.show()
    for box in results.boxes:
        cls_name = yolo.names[int(box.cls)]
        conf     = float(box.conf)
        print(f"  Detected: {cls_name} (conf={conf:.2f})")